# 4.2 Demand Prediction

In this notebook we implement Neural Networks (NNs) to predict taxi trip demand in chicago. Additionally, we compare the performances of the NNs across different complexity levels.

For NNs there are 2 "main" complexity interpretations:
- depth: number of hidden layers
- width: number of nodes per layer
- (more complex activation function) - maybe as an extra
- (more complex optimizer) - maybe as an extra

So we decide to test 3 different NN structures:

__baseline model__:
- hidden layers: 2
- nodes per layer: 64

__wider model__:
- hidden layers: 2
- nodes per layer: 128

__deeper model__:
- hidden layers: 4
- nodes per layer: 64

For better comparison, we will test all three architectures with the same shared configurations.

Before training any NN, we establish two simple **benchmarks** to contextualise the results:

1. **Historical-mean predictor** — for each hexagon × hour-of-day pair, predict the mean `trip_count` seen in the training set. Unseen combinations fall back to the global training mean. This captures the dominant demand pattern (location + time-of-day) without any learning.
2. **Ridge regression** — a linear model fit on the same scaled feature matrix. Trained on log₁⁺ demand and back-transformed at evaluation time (same target encoding as the NNs). This shows how much a linear model can do before adding any non-linearity or depth.

In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Import packages                         #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

import pandas as pd
# import matplotlib.pyplot as plt

# modeling
import copy
import random
import types
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score as _r2

# model visualization
from torchinfo import summary

# reset working dir
import os
from pathlib import Path


In [2]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Reset working directory                 #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

import __main__
_nb = getattr(__main__, "__vsc_ipynb_file__", None) or os.environ.get("JPY_SESSION_NAME")
_start = Path(_nb).resolve().parent if _nb else Path.cwd()
os.chdir(next(p for p in [_start, *_start.parents] if (p / "pyproject.toml").exists()))
print(f"Working directory: {os.getcwd()}")

Working directory: /Users/anthony/Documents/Dokumente – MacBook Pro von Anthony/UNI/AAA/AAA_TA_2026


In [3]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Load data                               #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

data = pd.read_parquet("data/aggregated/hexagon/demand_hex_1h_low.parquet")

In [30]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Data Information                        #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

# How many times does each row repeat when ignoring the time bucket?
# bucket_index is also excluded as it is a numeric encoding of the time bucket.
cols_no_time = [c for c in data.columns if c not in ('time_bucket', 'bucket_index')]
counts = data[cols_no_time].value_counts()

n_total    = len(data)
n_unique   = len(counts)
n_repeated = int((counts > 1).sum())

print(f"Total rows                    : {n_total:>10,}")
print(f"Unique combinations (no time) : {n_unique:>10,}")
print(f"Combinations appearing >1x    : {n_repeated:>10,}")
print(f"Avg repetitions per combo     : {n_total / n_unique:>10.2f}x")
print(f"\nRepetition count distribution:")
dist = counts.value_counts().sort_index()
for k, v in dist.items():
    print(f"  appears {k:>3}x : {v:>8,} combinations")

# How often does each hexagon + hour-of-day combination appear?
hex_hour_counts = data.groupby(['pickup_h3_res6', 'hour_of_day']).size()

print(f"\n--- Hexagon × Hour-of-day ---")
print(f"Unique hex × hour combinations : {len(hex_hour_counts):>8,}")
print(f"Avg appearances per combo      : {hex_hour_counts.mean():>8.1f}x")
print(f"Min appearances                : {hex_hour_counts.min():>8,}")
print(f"Max appearances                : {hex_hour_counts.max():>8,}")

# Zero-demand rows
n_zero        = int((data['trip_count'] == 0).sum())
pct_zero      = 100.0 * n_zero / n_total
print(f"\n--- Demand ---")
print(f"Zero-demand rows               : {n_zero:>8,}  ({pct_zero:.1f}%)")
print(f"Non-zero-demand rows           : {n_total - n_zero:>8,}  ({100 - pct_zero:.1f}%)")

Total rows                    :    289,080
Unique combinations (no time) :    289,080
Combinations appearing >1x    :          0
Avg repetitions per combo     :       1.00x

Repetition count distribution:
  appears   1x :  289,080 combinations

--- Hexagon × Hour-of-day ---
Unique hex × hour combinations :      792
Avg appearances per combo      :    365.0x
Min appearances                :      365
Max appearances                :      365

--- Demand ---
Zero-demand rows               :   91,126  (31.5%)
Non-zero-demand rows           :  197,954  (68.5%)


In [5]:
data.head()

,time_bucket,bucket_index,pickup_h3_res6,area_type,trip_count,active_taxis,avg_idle_time,avg_trip_duration,avg_trip_distance,avg_fare,...,dist_to_nearest_train_station_km,dist_to_nearest_stadium_km,train_station_per_km2,restaurants_per_km2,bars_and_clubs_per_km2,hotels_per_km2,hospitals_per_km2,universities_per_km2,attractions_per_km2,poi_density_total_per_km2
0,2025-01-01,482136,862664197ffffff,residential,0,0,0.0,0.0,0.000000,0.000000,...,3.463863,5.190882,0.000000,0.412471,0.219985,0.000000,0.00000,0.000000,0.027498,0.659954
1,2025-01-01,482136,86266419fffffff,residential,0,0,0.0,0.0,0.000000,0.000000,...,0.770790,5.836479,0.055042,0.137605,0.055042,0.055042,0.00000,0.000000,0.027521,0.330253
2,2025-01-01,482136,8626641b7ffffff,residential,0,0,0.0,0.0,0.000000,0.000000,...,3.428506,4.626667,0.027470,0.247232,0.027470,0.000000,0.00000,0.000000,0.000000,0.302173
3,2025-01-01,482136,862664527ffffff,airport,3,3,0.0,1228.0,9.536667,25.916667,...,2.529989,4.162891,0.027462,0.631634,0.302086,0.000000,0.00000,0.000000,0.000000,0.961182
4,2025-01-01,482136,86266452fffffff,residential,0,0,0.0,0.0,0.000000,0.000000,...,2.695998,5.548927,0.054970,0.659643,0.137426,0.384792,0.05497,0.027485,0.027485,1.346772


In [51]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Shared Configs                          #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

# --- optimization ---
OPTIMIZER          = "adam"
LEARNING_RATE      = 5e-5
BATCH_SIZE         = 256
MAX_EPOCHS         = 300

# --- loss / output ---
LOSS               = "mse"     # demand is count data, should try mae and mse too
OUTPUT_UNITS       = 1
OUTPUT_ACTIVATION  = "softplus"        # non-negative expected count

# --- layer defaults ---
HIDDEN_ACTIVATION  = "relu"
WEIGHT_INIT        = "he_normal"

# --- early stopping ---
EARLY_STOPPING     = True
MONITOR            = "val_loss"
PATIENCE           = 10

# --- data handling ---
SPLIT              = "random"
SCALER_FIT_ON      = "train_only"

# --- hex embedding ---
HEX_EMBED_DIM      = 16               # learned embedding dim per hexagon

# --- device ---
device = "cuda" if torch.cuda.is_available() else "cpu"

# --- reproducibility ---
SEEDS              = (0, 1, 2, 3, 4)   # run each model across all seeds; report mean +/- std

# --- architectures (the ONLY thing that varies across the 3 models) ---
# format: (n_hidden_layers, width)
ARCH_BASELINE      = (2, 64)
ARCH_WIDER         = (2, 128)          # depth fixed, width up
ARCH_DEEPER        = (4, 64)           # width fixed, depth up
# expose as module-like object so model code can use config.XXX
config = types.SimpleNamespace(
    LOSS           = LOSS,
    LEARNING_RATE  = LEARNING_RATE,
    BATCH_SIZE     = BATCH_SIZE,
    MAX_EPOCHS     = MAX_EPOCHS,
    EARLY_STOPPING = EARLY_STOPPING,
    PATIENCE       = PATIENCE,
    OUTPUT_UNITS   = OUTPUT_UNITS,
    HEX_EMBED_DIM  = HEX_EMBED_DIM,
)

ARCH_NAMES = {ARCH_BASELINE: "baseline", ARCH_WIDER: "wide", ARCH_DEEPER: "deep"}

# per-arch best LRs from LR search; falls back to config.LEARNING_RATE if not set
BEST_LR = {}

# --- hp search (random search over regularization + architecture hyperparams) ---
LR_CANDIDATES      = [2e-5, 3e-5, 4e-5, 5e-5, 6e-5, 7e-5, 1e-4]
HP_LR_CANDIDATES       = LR_CANDIDATES
HP_SEARCH_SEEDS        = (0,)                        # 1 seed sufficient for ranking combos
HP_PATIENCE            = 10               # lower patience during search; final runs use PATIENCE=10
BEST_HP                = {}              # {arch: {lr}}


In [33]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Model + Training Utilities              #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

class DemandBaseline(nn.Module):
    def __init__(self, input_dim, n_layers, width, n_hex, embed_dim):
        super().__init__()
        self.hex_embed = nn.Embedding(n_hex, embed_dim)
        nn.init.normal_(self.hex_embed.weight, std=0.01)

        layers = []
        in_dim = input_dim + embed_dim
        for _ in range(n_layers):
            linear = nn.Linear(in_dim, width)
            nn.init.kaiming_normal_(linear.weight, nonlinearity="relu")
            nn.init.zeros_(linear.bias)
            layers += [linear, nn.ReLU()]
            in_dim = width
        out = nn.Linear(in_dim, config.OUTPUT_UNITS)
        nn.init.kaiming_normal_(out.weight, nonlinearity="relu")
        nn.init.zeros_(out.bias)
        layers += [out, nn.Softplus()]
        self.net = nn.Sequential(*layers)

    def forward(self, x, hex_idx):
        emb = self.hex_embed(hex_idx)          # (batch, embed_dim)
        x   = torch.cat([x, emb], dim=-1)      # (batch, input_dim + embed_dim)
        return self.net(x).squeeze(-1)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def train_model(arch, X_train, y_train, X_val, y_val,
                hex_train, hex_val,
                seed=0, device="cpu", verbose=True, lr=None,
                batch_size=None, embed_dim=None,
                patience=None):
    set_seed(seed)
    n_layers, width = arch
    _embed_dim  = embed_dim    if embed_dim    is not None else config.HEX_EMBED_DIM
    _batch_size = batch_size   if batch_size   is not None else config.BATCH_SIZE
    _patience   = patience   if patience   is not None else config.PATIENCE
    model = DemandBaseline(
        X_train.shape[1], n_layers, width, N_HEX, _embed_dim,
    ).to(device)

    _loss_name = getattr(config, "LOSS", "poisson")
    if _loss_name == "mse":
        loss_fn = nn.MSELoss()
    elif _loss_name == "mae":
        loss_fn = nn.L1Loss()
    else:
        loss_fn = nn.PoissonNLLLoss(log_input=False, full=False)
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr if lr is not None else config.LEARNING_RATE,
    )

    train_dl = DataLoader(
        TensorDataset(
            torch.as_tensor(X_train,   dtype=torch.float32),
            torch.as_tensor(hex_train, dtype=torch.long),
            torch.as_tensor(y_train,   dtype=torch.float32),
        ),
        batch_size=_batch_size,
        shuffle=True,
    )
    X_val_t   = torch.as_tensor(X_val,   dtype=torch.float32).to(device)
    hex_val_t = torch.as_tensor(hex_val, dtype=torch.long).to(device)
    y_val_t   = torch.as_tensor(y_val,   dtype=torch.float32).to(device)

    best_val, best_state, wait = float("inf"), None, 0
    epoch_width = len(str(config.MAX_EPOCHS))

    for epoch in range(config.MAX_EPOCHS):
        # ── train ──────────────────────────────────────────────────────────────
        model.train()
        running_loss, n_batches = 0.0, 0
        for xb, hb, yb in train_dl:
            xb, hb, yb = xb.to(device), hb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(xb, hb), yb)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            n_batches    += 1
        train_loss = running_loss / n_batches

        # ── validate ───────────────────────────────────────────────────────────
        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(model(X_val_t, hex_val_t), y_val_t).item()

        # ── early stopping ─────────────────────────────────────────────────────
        improved = val_loss < best_val
        if improved:
            best_val, best_state, wait = val_loss, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1

        if verbose:
            marker = " *" if improved else f" (no improvement {wait}/{_patience})"
            print(f"  epoch {epoch+1:{epoch_width}d}/{config.MAX_EPOCHS}"
                  f"  train={train_loss:.4f}  val={val_loss:.4f}{marker}")

        if config.EARLY_STOPPING and wait >= _patience:
            if verbose:
                print(f"  early stopping at epoch {epoch+1}")
            break

    model.load_state_dict(best_state)
    return model, best_val


In [34]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# HP Search Utility                       #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

def run_hp_search(arch, X_train, y_train, X_val, y_val,
                  hex_train, hex_val,
                  device='cpu',
                  lr_candidates=None, seeds=None):
    import itertools

    lr_candidates = lr_candidates or HP_LR_CANDIDATES
    seeds         = seeds         or HP_SEARCH_SEEDS
    arch_label         = ARCH_NAMES.get(arch, str(arch))

    combos = list(itertools.product(
        lr_candidates,
    ))

    print(f'HP Search — {arch_label}')
    print(f'{len(combos)} combos × {len(seeds)} seed(s)'
          f' = {len(combos) * len(seeds)} runs  |  patience={HP_PATIENCE}\n')
    print(f"{'#':>4}  {'lr':>8}  {'mean val':>10}  {'std':>8}")
    print('-' * 35)

    results = []
    for i, (lr,) in enumerate(combos, 1):
        val_losses = []
        for seed in seeds:
            _, val_loss = train_model(
                arch, X_train, y_train, X_val, y_val,
                hex_train, hex_val,
                seed=seed, device=device, verbose=False,
                lr=lr, patience=HP_PATIENCE,
            )
            val_losses.append(val_loss)
        mean_loss = float(np.mean(val_losses))
        std_loss  = float(np.std(val_losses))
        results.append({
            'lr': lr,
            'mean_val_loss': mean_loss, 'std_val_loss': std_loss,
        })
        print(f'{i:>4}  {lr:>8.0e}  {mean_loss:>10.4f}  {std_loss:>8.4f}')

    results_df = pd.DataFrame(results).sort_values('mean_val_loss').reset_index(drop=True)
    print(f'\n--- Top 5 ---')
    print(results_df.head(5).to_string(index=False))

    best = results_df.iloc[0]
    BEST_HP[arch] = {
        'lr': float(best['lr']),
    }
    print(f'\nBEST_HP[{arch_label}] = {BEST_HP[arch]}')
    return BEST_HP[arch]


## Model Architecture — Data Flow

```
One row of raw data
┌─────────────────┬──────────────────────────────┬─────────────┐
│ pickup_h3_res7  │  is_weekend, temp, rain, ...  │ trip_count  │
│ '872664190fff'  │  0,  12.3,  0.0,  ...        │      5      │
└────────┬────────┴───────────────┬───────────────┴──────┬──────┘
         │                       │                       │
         ▼                       ▼                       ▼
    hex_vocab              StandardScaler             target y
  '872664190fff'          [0.0, -0.3, ...]              5.0
       → 42
         │                       │
         ▼                       │
  Embedding table                │
  (600 hexagons × 16 dims)       │
  row 42: [0.12, -0.31, ...]     │
     (16 dims, learned)          │ (29 dims, scaled)
         │                       │
         └───────────┬───────────┘
                     ▼
               torch.cat(...)
          [0.12, -0.31, ..., 0.0, -0.3, ...]
                  (16 + 29 = 45 dims)
                     │
                     ▼
           ┌─────────────────────┐
           │  Linear(45 → 64)    │
           │  ReLU               │
           │  Linear(64 → 64)    │  ← baseline / deeper adds more blocks
           │  ReLU               │
           │  Linear(64 → 1)     │
           │  Softplus           │  ← keeps output ≥ 0 (trip count)
           └─────────────────────┘
                     │
                     ▼
              ŷ  (predicted trip count)
```

The embedding table starts with near-zero weights and is updated by backprop
alongside all other parameters — the network learns which hexagons are similar.


## Train / Val / Test Split

We split rows randomly into **70 % train / 15 % val / 15 % test** using
`train_test_split` with a fixed random state for reproducibility.

| Split | Share |
|-------|-------|
| Train | ~70 % |
| Val   | ~15 % |
| Test  | ~15 % |


In [35]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Train / Val / Test Split                #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

data['time_bucket'] = pd.to_datetime(data['time_bucket'], format='mixed')

# 70 / 15 / 15 random split
train_data, temp_data = train_test_split(data, test_size=0.30, random_state=42, shuffle=False)
val_data,   test_data = train_test_split(temp_data, test_size=0.50, random_state=42, shuffle=False)

train_data = train_data.reset_index(drop=True)
val_data   = val_data.reset_index(drop=True)
test_data  = test_data.reset_index(drop=True)

print(f"Train : {len(train_data):>9,} rows")
print(f"Val   : {len(val_data):>9,} rows")
print(f"Test  : {len(test_data):>9,} rows")


Train :   202,356 rows
Val   :    43,362 rows
Test  :    43,362 rows


## Feature Preparation

Drop leakage columns (trip-derived aggregates from the same time-bucket),
ID/index columns, categorical columns not yet encoded, and the target.  
Fit a `StandardScaler` on the **training set only** to avoid leakage into val/test.


In [36]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Feature Preparation                     #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

# trip-derived stats (same time-bucket → leakage) + categoricals not yet encoded
LEAKAGE_COLS = [
    'active_taxis', 'avg_idle_time', 'avg_trip_duration', 'avg_trip_distance',
    'avg_fare', 'avg_trip_total', 'avg_tip', 'tip_rate', 'share_cash_payment',
    'area_type', 'season',
]

# raw cyclic integers replaced by sin/cos encodings
RAW_CYCLIC_COLS = ['month', 'hour_of_day', 'day_of_week']

ID_COLS    = ['time_bucket', 'bucket_index', 'pickup_h3_res6']
TARGET_COL = 'trip_count'

FEATURE_COLS = [
    c for c in train_data.columns
    if c not in LEAKAGE_COLS + RAW_CYCLIC_COLS + ID_COLS + [TARGET_COL]
]
print(f"Features ({len(FEATURE_COLS)}): {FEATURE_COLS}")

scaler  = StandardScaler()
X_train = scaler.fit_transform(train_data[FEATURE_COLS].values)
X_val   = scaler.transform(val_data[FEATURE_COLS].values)
X_test  = scaler.transform(test_data[FEATURE_COLS].values)

y_train = np.log1p(train_data[TARGET_COL].values.astype(float))
y_val   = np.log1p(val_data[TARGET_COL].values.astype(float))
y_test  = test_data[TARGET_COL].values.astype(float)  # kept raw for metric reporting

# hex embedding indices — vocabulary built from training set
hex_vocab = {h: i for i, h in enumerate(sorted(train_data['pickup_h3_res6'].unique()))}
N_HEX     = len(hex_vocab)

hex_train = train_data['pickup_h3_res6'].map(hex_vocab).values
hex_val   = val_data['pickup_h3_res6'].map(hex_vocab).values
hex_test  = test_data['pickup_h3_res6'].map(hex_vocab).values

print(f"\nX_train : {X_train.shape}   y_train : {y_train.shape}")
print(f"X_val   : {X_val.shape}     y_val   : {y_val.shape}")
print(f"X_test  : {X_test.shape}    y_test  : {y_test.shape}")
print(f"\nUnique hexagons : {N_HEX}  |  embed dim : {HEX_EMBED_DIM}")


Features (29): ['is_weekend', 'is_rush_hour', 'is_holiday', 'hour_of_day_sin', 'hour_of_day_cos', 'day_of_week_sin', 'day_of_week_cos', 'month_sin', 'month_cos', 'temperature_2m', 'apparent_temperature', 'precipitation', 'rain', 'snowfall', 'wind_speed_10m', 'cloud_cover', 'is_day', 'area_km2', 'dist_to_nearest_airport_km', 'dist_to_nearest_train_station_km', 'dist_to_nearest_stadium_km', 'train_station_per_km2', 'restaurants_per_km2', 'bars_and_clubs_per_km2', 'hotels_per_km2', 'hospitals_per_km2', 'universities_per_km2', 'attractions_per_km2', 'poi_density_total_per_km2']

X_train : (202356, 29)   y_train : (202356,)
X_val   : (43362, 29)     y_val   : (43362,)
X_test  : (43362, 29)    y_test  : (43362,)

Unique hexagons : 33  |  embed dim : 16


## Benchmark Models

Before diving into the three NN architectures, we evaluate two simple non-neural baselines on the held-out test set:

1. **Historical-mean predictor** — for each hexagon × hour-of-day pair, predict the mean `trip_count` observed in the training split. Unseen combinations fall back to the global training mean.
2. **Ridge regression** — a linear model on the same scaled feature matrix (`X_train`). Trained on log₁⁺ demand and back-transformed with `expm1` at evaluation time, matching the NN target encoding.

These numbers set the floor: any NN that cannot beat Ridge is not adding value.

Metrics match the NN evaluation cells:
- **R²** — coefficient of determination
- **MAE** — mean absolute error (in raw trip counts)
- **RMSE** — root mean squared error
- **NRMSE** — RMSE normalised by mean actual demand (scale-free)

In [37]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Benchmark Models                        #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

zero_demand_count = int((y_test == 0).sum())
zero_demand_pct   = 100.0 * zero_demand_count / len(y_test)
mean_actual       = float(y_test.mean())
ss_tot            = float(np.sum((y_test - mean_actual) ** 2))

print(f"Zero-demand rows : {zero_demand_count:,} / {len(y_test):,} ({zero_demand_pct:.1f}%)")
print(f"Mean actual test demand : {mean_actual:.4f}\n")


def _eval_benchmark(label, preds):
    mae   = float(np.mean(np.abs(preds - y_test)))
    rmse  = float(np.sqrt(np.mean((preds - y_test) ** 2)))
    nrmse = rmse / mean_actual if mean_actual > 0 else float("nan")
    r2    = 1.0 - float(np.sum((preds - y_test) ** 2)) / ss_tot
    print(f"{label:<35s}  R²={r2:.4f}  MAE={mae:.4f}  RMSE={rmse:.4f}  NRMSE={nrmse:.4f}")
    return dict(r2=round(r2, 6), mae=round(mae, 6), rmse=round(rmse, 6), nrmse=round(nrmse, 6))


# ── 1. Historical-mean predictor ───────────────────────────────────────────────
mean_table = (
    train_data.groupby(['pickup_h3_res6', 'hour_of_day'])['trip_count']
    .mean()
    .rename('pred_mean')
)
global_mean = float(train_data['trip_count'].mean())

test_lookup      = test_data[['pickup_h3_res6', 'hour_of_day']].copy()
test_lookup      = test_lookup.join(mean_table, on=['pickup_h3_res6', 'hour_of_day'])
hist_mean_preds  = test_lookup['pred_mean'].fillna(global_mean).values

bm_hist = _eval_benchmark("Historical mean (hex × hour)", hist_mean_preds)

# ── 2. Ridge regression ────────────────────────────────────────────────────────
ridge        = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)            # y_train is log1p-transformed
ridge_preds  = np.expm1(ridge.predict(X_test))
ridge_preds  = np.maximum(ridge_preds, 0.0)   # clamp to non-negative

bm_ridge = _eval_benchmark("Ridge regression", ridge_preds)


Zero-demand rows : 15,505 / 43,362 (35.8%)
Mean actual test demand : 17.5691

Historical mean (hex × hour)         R²=0.8329  MAE=7.2371  RMSE=22.9434  NRMSE=1.3059
Ridge regression                     R²=0.6284  MAE=10.8343  RMSE=34.2168  NRMSE=1.9476


## Baseline Model — Training

Run `ARCH_BASELINE = (2 hidden layers, 64 units)` across all seeds and report
the mean ± std validation loss.


In [38]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Baseline Model — Visualization          #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

input_dim = X_train.shape[1]
n_layers, width = ARCH_BASELINE

viz_model = DemandBaseline(input_dim, n_layers, width, N_HEX, HEX_EMBED_DIM)
x_dummy   = torch.zeros(1, input_dim)
h_dummy   = torch.zeros(1, dtype=torch.long)
summary(viz_model, input_data=(x_dummy, h_dummy), col_names=["input_size", "output_size", "num_params"], verbose=1)


Layer (type:depth-idx)                   Input Shape               Output Shape              Param #
DemandBaseline                           [1, 29]                   [1]                       --
├─Embedding: 1-1                         [1]                       [1, 16]                   528
├─Sequential: 1-2                        [1, 45]                   [1, 1]                    --
│    └─Linear: 2-1                       [1, 45]                   [1, 64]                   2,944
│    └─ReLU: 2-2                         [1, 64]                   [1, 64]                   --
│    └─Linear: 2-3                       [1, 64]                   [1, 64]                   4,160
│    └─ReLU: 2-4                         [1, 64]                   [1, 64]                   --
│    └─Linear: 2-5                       [1, 64]                   [1, 1]                    65
│    └─Softplus: 2-6                     [1, 1]                    [1, 1]                    --
Total params: 7,697
Trainabl

Layer (type:depth-idx)                   Input Shape               Output Shape              Param #
DemandBaseline                           [1, 29]                   [1]                       --
├─Embedding: 1-1                         [1]                       [1, 16]                   528
├─Sequential: 1-2                        [1, 45]                   [1, 1]                    --
│    └─Linear: 2-1                       [1, 45]                   [1, 64]                   2,944
│    └─ReLU: 2-2                         [1, 64]                   [1, 64]                   --
│    └─Linear: 2-3                       [1, 64]                   [1, 64]                   4,160
│    └─ReLU: 2-4                         [1, 64]                   [1, 64]                   --
│    └─Linear: 2-5                       [1, 64]                   [1, 1]                    65
│    └─Softplus: 2-6                     [1, 1]                    [1, 1]                    --
Total params: 7,697
Trainabl

In [52]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Baseline Model — Training               #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

print(f"Device: {device}\n")

baseline_val_losses  = []
baseline_models      = []

for seed in SEEDS:
    model, val_loss = train_model(
        ARCH_BASELINE, X_train, y_train, X_val, y_val,
        hex_train, hex_val,
        seed=seed, device=device,
        lr=BEST_LR.get(ARCH_BASELINE, config.LEARNING_RATE),
    )
    baseline_val_losses.append(val_loss)
    baseline_models.append(model)
    print(f"  seed={seed}  val_loss={val_loss:.4f}")

print(f"\nBaseline  val loss:  {np.mean(baseline_val_losses):.4f} ± {np.std(baseline_val_losses):.4f}")


Device: cpu

  epoch   1/300  train=1.4389  val=0.8484 *
  epoch   2/300  train=0.5498  val=0.4100 *
  epoch   3/300  train=0.3235  val=0.3451 *
  epoch   4/300  train=0.2716  val=0.3105 *
  epoch   5/300  train=0.2459  val=0.2949 *
  epoch   6/300  train=0.2306  val=0.2863 *
  epoch   7/300  train=0.2204  val=0.2742 *
  epoch   8/300  train=0.2131  val=0.2687 *
  epoch   9/300  train=0.2071  val=0.2616 *
  epoch  10/300  train=0.2022  val=0.2586 *
  epoch  11/300  train=0.1981  val=0.2552 *
  epoch  12/300  train=0.1945  val=0.2501 *
  epoch  13/300  train=0.1913  val=0.2455 *
  epoch  14/300  train=0.1885  val=0.2423 *
  epoch  15/300  train=0.1860  val=0.2397 *
  epoch  16/300  train=0.1838  val=0.2360 *
  epoch  17/300  train=0.1817  val=0.2344 *
  epoch  18/300  train=0.1798  val=0.2288 *
  epoch  19/300  train=0.1781  val=0.2275 *
  epoch  20/300  train=0.1764  val=0.2242 *
  epoch  21/300  train=0.1750  val=0.2235 *
  epoch  22/300  train=0.1736  val=0.2202 *
  epoch  23/300  tr

## Baseline Model — Evaluation

Evaluate on the held-out **test set** using the best-checkpoint model from each
seed, then report mean ± std across seeds.

Metrics:
- **Zero-demand rows** — share of test rows with `trip_count = 0` (context for inflated R²)
- **Mean Actual Test Demand** — average true trip count in the test set
- **R²** — coefficient of determination (share of variance explained)
- **MAE** — mean absolute error (interpretable in trip counts)
- **RMSE** — root mean squared error (penalises large misses more)
- **NRMSE** — RMSE normalised by mean actual demand (scale-free)

In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Baseline Model — Evaluation             #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

import datetime

X_test_t   = torch.as_tensor(X_test,   dtype=torch.float32).to(device)
hex_test_t = torch.as_tensor(hex_test, dtype=torch.long).to(device)

run_results  = []
r2_scores,     mae_scores,     rmse_scores,     nrmse_scores     = [], [], [], []
r2_log_scores, mae_log_scores, rmse_log_scores, nrmse_log_scores = [], [], [], []
correct_zero_counts, false_zero_counts = [], []
n_layers, width = ARCH_BASELINE

zero_demand_count = int((y_test == 0).sum())
zero_demand_pct   = 100.0 * zero_demand_count / len(y_test)
mean_actual       = float(y_test.mean())
ss_tot            = float(np.sum((y_test - mean_actual) ** 2))

y_test_log  = np.log1p(y_test)
mean_log    = float(y_test_log.mean())
ss_tot_log  = float(np.sum((y_test_log - mean_log) ** 2))

print(f"Zero-demand rows: {zero_demand_count:,} / {len(y_test):,} ({zero_demand_pct:.1f}%)")
print(f"Mean actual test demand : {mean_actual:.4f}\n")

for seed, model in zip(SEEDS, baseline_models):
    model.eval()
    with torch.no_grad():
        preds_log = model(X_test_t, hex_test_t).cpu().numpy().flatten()
    preds = np.expm1(preds_log)

    mae   = float(np.mean(np.abs(preds - y_test)))
    rmse  = float(np.sqrt(np.mean((preds - y_test) ** 2)))
    nrmse = rmse / mean_actual if mean_actual > 0 else float("nan")
    r2    = 1.0 - float(np.sum((preds - y_test) ** 2)) / ss_tot

    mae_log   = float(np.mean(np.abs(preds_log - y_test_log)))
    rmse_log  = float(np.sqrt(np.mean((preds_log - y_test_log) ** 2)))
    nrmse_log = rmse_log / mean_log if mean_log > 0 else float("nan")
    r2_log    = 1.0 - float(np.sum((preds_log - y_test_log) ** 2)) / ss_tot_log

    zero_actual  = (y_test == 0)
    zero_pred    = (preds < 0.5)
    correct_zero = int(np.sum(zero_actual & zero_pred))
    false_zero   = int(np.sum(~zero_actual & zero_pred))

    r2_scores.append(r2);         mae_scores.append(mae)
    rmse_scores.append(rmse);     nrmse_scores.append(nrmse)
    r2_log_scores.append(r2_log); mae_log_scores.append(mae_log)
    rmse_log_scores.append(rmse_log); nrmse_log_scores.append(nrmse_log)
    correct_zero_counts.append(correct_zero)
    false_zero_counts.append(false_zero)

    print(f"  seed={seed}"
          f"  R\u00b2={r2:.4f}  MAE={mae:.4f}  NRMSE={nrmse:.4f}"
          f"  | log  R\u00b2={r2_log:.4f}  MAE={mae_log:.4f}  NRMSE={nrmse_log:.4f}"
          f"  | zeros {correct_zero:,}/{zero_demand_count:,}  false={false_zero:,}")

    run_results.append({
        "timestamp"         : datetime.datetime.now().isoformat(timespec="seconds"),
        "model"             : ARCH_NAMES.get((n_layers, width), "unknown"),
        "n_layers"          : n_layers,
        "width"             : width,
        "learning_rate"     : BEST_LR.get(ARCH_BASELINE, config.LEARNING_RATE),
        "batch_size"        : BATCH_SIZE,
        "seed"              : seed,
        "val_loss"          : baseline_val_losses[seed],
        "zero_demand_count" : zero_demand_count,
        "mean_actual"       : round(mean_actual, 6),
        "r2"                : round(r2,       6),
        "mae"               : round(mae,      6),
        "rmse"              : round(rmse,     6),
        "nrmse"             : round(nrmse,    6),
        "r2_log"            : round(r2_log,   6),
        "mae_log"           : round(mae_log,  6),
        "rmse_log"          : round(rmse_log, 6),
        "nrmse_log"         : round(nrmse_log,6),
        "correct_zero"      : correct_zero,
        "false_zero"        : false_zero,
    })

arch_label = ARCH_NAMES.get((n_layers, width), "model")
print(f"\n--- original scale ---")
print(f"{arch_label.capitalize()}  R\u00b2    : {np.mean(r2_scores):.4f} \u00b1 {np.std(r2_scores):.4f}")
print(f"{arch_label.capitalize()}  MAE   : {np.mean(mae_scores):.4f} \u00b1 {np.std(mae_scores):.4f}")
print(f"{arch_label.capitalize()}  RMSE  : {np.mean(rmse_scores):.4f} \u00b1 {np.std(rmse_scores):.4f}")
print(f"{arch_label.capitalize()}  NRMSE : {np.mean(nrmse_scores):.4f} \u00b1 {np.std(nrmse_scores):.4f}")
print(f"\n--- log1p scale ---")
print(f"{arch_label.capitalize()}  R\u00b2    : {np.mean(r2_log_scores):.4f} \u00b1 {np.std(r2_log_scores):.4f}")
print(f"{arch_label.capitalize()}  MAE   : {np.mean(mae_log_scores):.4f} \u00b1 {np.std(mae_log_scores):.4f}")
print(f"{arch_label.capitalize()}  RMSE  : {np.mean(rmse_log_scores):.4f} \u00b1 {np.std(rmse_log_scores):.4f}")
print(f"{arch_label.capitalize()}  NRMSE : {np.mean(nrmse_log_scores):.4f} \u00b1 {np.std(nrmse_log_scores):.4f}")
print(f"\n--- zero-demand ---")
print(f"{arch_label.capitalize()}  Correct-zero : {np.mean(correct_zero_counts):.1f} \u00b1 {np.std(correct_zero_counts):.1f}  (out of {zero_demand_count:,})")
print(f"{arch_label.capitalize()}  False-zero   : {np.mean(false_zero_counts):.1f} \u00b1 {np.std(false_zero_counts):.1f}  (predicted 0 when demand > 0)")

Tuned Baseline  R²=0.9313 ± 0.0020  MAE=4.4563 ± 0.0101  NRMSE=0.7787 ± 0.0113


In [46]:
results_df

,timestamp,model,n_layers,width,learning_rate,batch_size,seed,val_loss,zero_demand_count,mean_actual,r2,mae,rmse,nrmse
0,2026-06-28T16:44:28,baseline,2,64,0.00005,1024,0,0.206478,15505,17.569093,0.767528,7.494371,27.062424,1.540343
1,2026-06-28T16:44:28,baseline,2,64,0.00005,1024,1,0.208102,15505,17.569093,0.764689,7.582091,27.227162,1.549719
2,2026-06-28T16:44:28,baseline,2,64,0.00005,1024,2,0.207375,15505,17.569093,0.756782,7.727719,27.680854,1.575543
3,2026-06-28T16:44:28,baseline,2,64,0.00005,1024,3,0.210005,15505,17.569093,0.763795,7.549308,27.278847,1.552661
4,2026-06-28T16:44:28,baseline,2,64,0.00005,1024,4,0.205678,15505,17.569093,0.764083,7.399505,27.262207,1.551714


In [47]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Save Results                            #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

RESULTS_PATH = "data/results/nn_results.csv"
os.makedirs(os.path.dirname(RESULTS_PATH), exist_ok=True)

results_df = pd.DataFrame(run_results)

if os.path.exists(RESULTS_PATH):
    results_df.to_csv(RESULTS_PATH, mode="a", header=False, index=False)
    print(f"Appended {len(results_df)} rows to {RESULTS_PATH}")
else:
    results_df.to_csv(RESULTS_PATH, index=False)
    print(f"Created {RESULTS_PATH} with {len(results_df)} rows")

print(pd.read_csv(RESULTS_PATH).tail(len(results_df)).to_string(index=False))


Appended 5 rows to data/results/nn_results.csv
          timestamp    model  n_layers  width  learning_rate  batch_size  seed  val_loss  zero_demand_count  mean_actual       r2      mae      rmse    nrmse
2026-06-28T16:44:28 baseline         2     64        0.00005        1024     0  0.206478            15505.0    17.569093 0.767528 7.494371 27.062424 1.540343
2026-06-28T16:44:28 baseline         2     64        0.00005        1024     1  0.208102            15505.0    17.569093 0.764689 7.582091 27.227162 1.549719
2026-06-28T16:44:28 baseline         2     64        0.00005        1024     2  0.207375            15505.0    17.569093 0.756782 7.727719 27.680854 1.575543
2026-06-28T16:44:28 baseline         2     64        0.00005        1024     3  0.210005            15505.0    17.569093 0.763795 7.549308 27.278847 1.552661
2026-06-28T16:44:28 baseline         2     64        0.00005        1024     4  0.205678            15505.0    17.569093 0.764083 7.399505 27.262207 1.551714


## Representation Probing — Baseline Model

**Question**: can the model infer the specific calendar **date** from its input
features? If yes, random train/test splitting is unsafe — same-date rows
(identical weather, same day-of-week) end up in both train and test, and the
model can exploit that fingerprint rather than learning to generalise.

**Protocol**

1. Register a forward hook on the **last ReLU** (64-dim layer) and extract
   activations for all train and test rows.
2. Derive sequential date targets from `time_bucket` (all strip the hour so
   probing is purely about which calendar period it is):
   - `date_index` — days since first date in dataset
   - `week_index` — weeks since first date in dataset
3. For each target fit a `Ridge` linear probe and compare three probes:

   | Probe | What it measures |
   |---|---|
   | **Cyclic-only probe** | Floor: sin/cos time features — cannot distinguish e.g. week 1 from week 52 |
   | **Full raw-X probe** | Ceiling: all 29 scaled inputs including weather |
   | **Repr probe** | What the 64-dim hidden activations encode |

R²(repr) − R²(cyclic floor) > 0.10 → model encodes specific dates → chronological split needed.

**Sanity check**: re-running with shuffled labels must collapse R² to ~0.

In [16]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Representation-Probing Test             #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score as _r2

# Cyclic feature column names (sin/cos encodings already in X)
CYCLIC_COLS = [c for c in FEATURE_COLS
               if any(c.startswith(p) for p in
                      ('hour_of_day_', 'day_of_week_', 'month_'))]
CYCLIC_IDX  = [FEATURE_COLS.index(c) for c in CYCLIC_COLS]

# Probe targets: days / weeks since first date in dataset (no time component)
_min_date  = data['time_bucket'].dt.normalize().min()
_days_tr   = (train_data['time_bucket'].dt.normalize() - _min_date).dt.days
_days_te   = (test_data['time_bucket'].dt.normalize()  - _min_date).dt.days

PROBE_TARGETS = {
    'date_index' : (_days_tr.values.astype(float),       _days_te.values.astype(float)),
    'week_index' : ((_days_tr // 7).values.astype(float), (_days_te // 7).values.astype(float)),
}


def extract_hidden_reps(model, X, hex_idx, device, batch_size=4096):
    model.eval()
    captured = []

    last_relu = None
    for layer in model.net:
        if isinstance(layer, nn.ReLU):
            last_relu = layer

    def _hook(_, __, output):
        captured.append(output.detach().cpu())

    handle = last_relu.register_forward_hook(_hook)
    X_t = torch.as_tensor(X,       dtype=torch.float32)
    h_t = torch.as_tensor(hex_idx, dtype=torch.long)

    with torch.no_grad():
        for start in range(0, X.shape[0], batch_size):
            model(
                X_t[start : start + batch_size].to(device),
                h_t[start : start + batch_size].to(device),
            )

    handle.remove()
    return np.vstack([t.numpy() for t in captured])


# Use seed-0 baseline model (trained in cell above)
probe_model = baseline_models[0]

print('Extracting hidden representations ...')
reps_train = extract_hidden_reps(probe_model, X_train, hex_train, device)
reps_test  = extract_hidden_reps(probe_model, X_test,  hex_test,  device)
print(f'  train reps: {reps_train.shape}   test reps: {reps_test.shape}\n')

rng = np.random.default_rng(42)

for target_name, (y_tr, y_te) in PROBE_TARGETS.items():
    r2_cyclic = _r2(y_te, Ridge(alpha=1.0).fit(X_train[:, CYCLIC_IDX], y_tr)
                                            .predict(X_test[:, CYCLIC_IDX]))
    r2_raw    = _r2(y_te, Ridge(alpha=1.0).fit(X_train, y_tr).predict(X_test))
    r2_repr   = _r2(y_te, Ridge(alpha=1.0).fit(reps_train, y_tr).predict(reps_test))
    r2_shuf   = _r2(y_te, Ridge(alpha=1.0).fit(reps_train, rng.permutation(y_tr))
                                            .predict(reps_test))

    above_floor = r2_repr - r2_cyclic
    verdict = ('-> chronological split recommended'
               if above_floor > 0.10 else '-> random split acceptable')

    print(f'Target: {target_name}')
    print(f'  Cyclic-only probe  R2 = {r2_cyclic:.4f}  <- floor')
    print(f'  Full raw-X probe   R2 = {r2_raw:.4f}  <- ceiling')
    print(f'  Repr probe         R2 = {r2_repr:.4f}')
    print(f'  Shuffled (sanity)  R2 = {r2_shuf:.4f}  <- should be ~0')
    print(f'  Delta (repr - floor) = {above_floor:+.4f}  {verdict}')
    print()


Extracting hidden representations ...
  train reps: (202356, 64)   test reps: (43362, 64)

Target: date_index
  Cyclic-only probe  R2 = -242.8449  <- floor
  Full raw-X probe   R2 = -239.6795  <- ceiling
  Repr probe         R2 = -295.8213
  Shuffled (sanity)  R2 = -175.8046  <- should be ~0
  Delta (repr - floor) = -52.9764  -> random split acceptable

Target: week_index
  Cyclic-only probe  R2 = -238.7591  <- floor
  Full raw-X probe   R2 = -235.5967  <- ceiling
  Repr probe         R2 = -291.3155
  Shuffled (sanity)  R2 = -173.7902  <- should be ~0
  Delta (repr - floor) = -52.5564  -> random split acceptable



## Embed Dimension Search

Since `embed_dim` is independent of architecture depth and width, we search it once on
the baseline and fix the winner globally for all three models.

Candidates: `[8, 16, 32]` — 3 runs total (1 seed, early stopping).

In [17]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Embed Dimension Search                  #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

EMBED_CANDIDATES = [8, 16, 32]

print(f"Embed search — ARCH_BASELINE, seed=0, patience={HP_PATIENCE}\n")
embed_results = []
for emb in EMBED_CANDIDATES:
    _, val_loss = train_model(
        ARCH_BASELINE, X_train, y_train, X_val, y_val,
        hex_train, hex_val,
        seed=0, device=device, verbose=False,
        lr=config.LEARNING_RATE, embed_dim=emb, patience=HP_PATIENCE,
    )
    embed_results.append((emb, val_loss))
    print(f"  embed_dim={emb:>3}  val_loss={val_loss:.4f}")

best_embed_dim = min(embed_results, key=lambda x: x[1])[0]
HEX_EMBED_DIM        = best_embed_dim
config.HEX_EMBED_DIM = best_embed_dim
print(f"\nBest embed_dim: {HEX_EMBED_DIM}  →  set as global HEX_EMBED_DIM")


Embed search — ARCH_BASELINE, seed=0, patience=10

  embed_dim=  8  val_loss=0.2073
  embed_dim= 16  val_loss=0.1962
  embed_dim= 32  val_loss=0.1948

Best embed_dim: 32  →  set as global HEX_EMBED_DIM


## Hyperparameter Search — Baseline Model

Grid search over 4 hyperparameters using the best LR from the LR search above.

| Hyperparameter | Candidates |
|---|---|
| `lr`           | 2e-5, 3e-5, 4e-5, 5e-5, 6e-5, 7e-5, 1e-4 |

Full grid: 4 × 4 × 3 = **48 combinations** × 1 seed = **48 training runs**.  
The best combination is stored in `BEST_HP[ARCH_BASELINE]`.


In [48]:
run_hp_search(
    ARCH_BASELINE,
    X_train, y_train, X_val, y_val,
    hex_train, hex_val,
    device=device,
)

HP Search — baseline
7 combos × 1 seed(s) = 7 runs  |  patience=10

   #        lr    mean val       std
-----------------------------------
   1     2e-05      0.2069    0.0000
   2     3e-05      0.2019    0.0000
   3     4e-05      0.1985    0.0000
   4     5e-05      0.1962    0.0000
   5     6e-05      0.1945    0.0000
   6     7e-05      0.1960    0.0000
   7     1e-04      0.1967    0.0000

--- Top 5 ---
     lr  mean_val_loss  std_val_loss
0.00006       0.194548           0.0
0.00007       0.196011           0.0
0.00005       0.196220           0.0
0.00010       0.196666           0.0
0.00004       0.198543           0.0

BEST_HP[baseline] = {'lr': 6e-05}


{'lr': 6e-05}

## Tuned Baseline — Training

Re-train the baseline architecture with the best hyperparameters found above,
across all `SEEDS` for a stable mean ± std estimate.


In [49]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Tuned Baseline — Training               #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

hp = BEST_HP[ARCH_BASELINE]
print(f"Best HP: {hp}\n")

tuned_val_losses = []
tuned_models     = []

for seed in SEEDS:
    model, val_loss = train_model(
        ARCH_BASELINE, X_train, y_train, X_val, y_val,
        hex_train, hex_val,
        seed=seed, device=device,
        lr=hp["lr"],
    )
    tuned_val_losses.append(val_loss)
    tuned_models.append(model)
    print(f"  seed={seed}  val_loss={val_loss:.4f}")

print(f"\nTuned Baseline  val loss:  "
      f"{np.mean(tuned_val_losses):.4f} ± {np.std(tuned_val_losses):.4f}")


Best HP: {'lr': 6e-05}

  epoch   1/100  train=1.3283  val=0.7095 *
  epoch   2/100  train=0.4476  val=0.3743 *
  epoch   3/100  train=0.2939  val=0.3257 *
  epoch   4/100  train=0.2536  val=0.2967 *
  epoch   5/100  train=0.2332  val=0.2847 *
  epoch   6/100  train=0.2209  val=0.2778 *
  epoch   7/100  train=0.2122  val=0.2658 *
  epoch   8/100  train=0.2057  val=0.2604 *
  epoch   9/100  train=0.2002  val=0.2538 *
  epoch  10/100  train=0.1957  val=0.2505 *
  epoch  11/100  train=0.1918  val=0.2478 *
  epoch  12/100  train=0.1885  val=0.2419 *
  epoch  13/100  train=0.1855  val=0.2375 *
  epoch  14/100  train=0.1829  val=0.2340 *
  epoch  15/100  train=0.1805  val=0.2325 *
  epoch  16/100  train=0.1784  val=0.2279 *
  epoch  17/100  train=0.1764  val=0.2268 *
  epoch  18/100  train=0.1747  val=0.2217 *
  epoch  19/100  train=0.1731  val=0.2209 *
  epoch  20/100  train=0.1717  val=0.2176 *
  epoch  21/100  train=0.1704  val=0.2167 *
  epoch  22/100  train=0.1692  val=0.2146 *
  epoch 

## Tuned Baseline — Evaluation

Evaluate on the held-out **test set** and compare against the untuned baseline.


In [55]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Tuned Baseline — Evaluation             #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

import datetime

X_test_t   = torch.as_tensor(X_test,   dtype=torch.float32).to(device)
hex_test_t = torch.as_tensor(hex_test, dtype=torch.long).to(device)

run_results  = []
r2_scores,     mae_scores,     rmse_scores,     nrmse_scores     = [], [], [], []
r2_log_scores, mae_log_scores, rmse_log_scores, nrmse_log_scores = [], [], [], []
correct_zero_counts, false_zero_counts = [], []
n_layers, width = ARCH_BASELINE

zero_demand_count = int((y_test == 0).sum())
zero_demand_pct   = 100.0 * zero_demand_count / len(y_test)
mean_actual       = float(y_test.mean())
ss_tot            = float(np.sum((y_test - mean_actual) ** 2))

y_test_log  = np.log1p(y_test)
mean_log    = float(y_test_log.mean())
ss_tot_log  = float(np.sum((y_test_log - mean_log) ** 2))

print(f"Zero-demand rows: {zero_demand_count:,} / {len(y_test):,} ({zero_demand_pct:.1f}%)")
print(f"Mean actual test demand : {mean_actual:.4f}\n")

for seed, model in zip(SEEDS, tuned_models):
    model.eval()
    with torch.no_grad():
        preds_log = model(X_test_t, hex_test_t).cpu().numpy().flatten()
    preds = np.expm1(preds_log)

    mae   = float(np.mean(np.abs(preds - y_test)))
    rmse  = float(np.sqrt(np.mean((preds - y_test) ** 2)))
    nrmse = rmse / mean_actual if mean_actual > 0 else float("nan")
    r2    = 1.0 - float(np.sum((preds - y_test) ** 2)) / ss_tot

    mae_log   = float(np.mean(np.abs(preds_log - y_test_log)))
    rmse_log  = float(np.sqrt(np.mean((preds_log - y_test_log) ** 2)))
    nrmse_log = rmse_log / mean_log if mean_log > 0 else float("nan")
    r2_log    = 1.0 - float(np.sum((preds_log - y_test_log) ** 2)) / ss_tot_log

    zero_actual  = (y_test == 0)
    zero_pred    = (preds < 0.5)
    correct_zero = int(np.sum(zero_actual & zero_pred))
    false_zero   = int(np.sum(~zero_actual & zero_pred))

    r2_scores.append(r2);         mae_scores.append(mae)
    rmse_scores.append(rmse);     nrmse_scores.append(nrmse)
    r2_log_scores.append(r2_log); mae_log_scores.append(mae_log)
    rmse_log_scores.append(rmse_log); nrmse_log_scores.append(nrmse_log)
    correct_zero_counts.append(correct_zero)
    false_zero_counts.append(false_zero)

    print(f"  seed={seed}"
          f"  R\u00b2={r2:.4f}  MAE={mae:.4f}  NRMSE={nrmse:.4f}"
          f"  | log  R\u00b2={r2_log:.4f}  MAE={mae_log:.4f}  NRMSE={nrmse_log:.4f}"
          f"  | zeros {correct_zero:,}/{zero_demand_count:,}  false={false_zero:,}")

    run_results.append({
        "timestamp"         : datetime.datetime.now().isoformat(timespec="seconds"),
        "model"             : "baseline_tuned",
        "n_layers"          : n_layers,
        "width"             : width,
        "learning_rate"     : BEST_LR.get(ARCH_BASELINE, config.LEARNING_RATE),
        "batch_size"        : BATCH_SIZE,
        "seed"              : seed,
        "val_loss"          : tuned_val_losses[seed],
        "zero_demand_count" : zero_demand_count,
        "mean_actual"       : mean_actual,
        "r2"                : r2,
        "mae"               : mae,
        "rmse"              : rmse,
        "nrmse"             : nrmse,
        "r2_log"            : round(r2_log,   6),
        "mae_log"           : round(mae_log,  6),
        "rmse_log"          : round(rmse_log, 6),
        "nrmse_log"         : round(nrmse_log,6),
        "correct_zero"      : correct_zero,
        "false_zero"        : false_zero,
    })

print(f"\n--- original scale ---")
print(f"Tuned Baseline  R\u00b2    : {np.mean(r2_scores):.4f} \u00b1 {np.std(r2_scores):.4f}")
print(f"Tuned Baseline  MAE   : {np.mean(mae_scores):.4f} \u00b1 {np.std(mae_scores):.4f}")
print(f"Tuned Baseline  RMSE  : {np.mean(rmse_scores):.4f} \u00b1 {np.std(rmse_scores):.4f}")
print(f"Tuned Baseline  NRMSE : {np.mean(nrmse_scores):.4f} \u00b1 {np.std(nrmse_scores):.4f}")
print(f"\n--- log1p scale ---")
print(f"Tuned Baseline  R\u00b2    : {np.mean(r2_log_scores):.4f} \u00b1 {np.std(r2_log_scores):.4f}")
print(f"Tuned Baseline  MAE   : {np.mean(mae_log_scores):.4f} \u00b1 {np.std(mae_log_scores):.4f}")
print(f"Tuned Baseline  RMSE  : {np.mean(rmse_log_scores):.4f} \u00b1 {np.std(rmse_log_scores):.4f}")
print(f"Tuned Baseline  NRMSE : {np.mean(nrmse_log_scores):.4f} \u00b1 {np.std(nrmse_log_scores):.4f}")
print(f"\n--- zero-demand ---")
print(f"Tuned Baseline  Correct-zero : {np.mean(correct_zero_counts):.1f} \u00b1 {np.std(correct_zero_counts):.1f}  (out of {zero_demand_count:,})")
print(f"Tuned Baseline  False-zero   : {np.mean(false_zero_counts):.1f} \u00b1 {np.std(false_zero_counts):.1f}  (predicted 0 when demand > 0)")

Zero-demand rows: 15,505 / 43,362 (35.8%)
Mean actual test demand : 17.5691

  seed=0  R²=0.7999  MAE=6.9785  NRMSE=1.4292  | log  R²=0.8930  MAE=0.3579  NRMSE=0.3559  | zeros 11,703/15,505  false=1,947
  seed=1  R²=0.7710  MAE=7.4515  NRMSE=1.5287  | log  R²=0.8888  MAE=0.3662  NRMSE=0.3627  | zeros 11,804/15,505  false=2,008
  seed=2  R²=0.7543  MAE=7.6604  NRMSE=1.5834  | log  R²=0.8859  MAE=0.3712  NRMSE=0.3674  | zeros 11,729/15,505  false=1,966
  seed=3  R²=0.7561  MAE=7.5321  NRMSE=1.5776  | log  R²=0.8885  MAE=0.3666  NRMSE=0.3632  | zeros 11,440/15,505  false=1,760
  seed=4  R²=0.7687  MAE=7.2955  NRMSE=1.5364  | log  R²=0.8907  MAE=0.3621  NRMSE=0.3597  | zeros 11,650/15,505  false=1,862

--- original scale ---
Tuned Baseline  R²    : 0.7700 ± 0.0163
Tuned Baseline  MAE   : 7.3836 ± 0.2346
Tuned Baseline  RMSE  : 26.8994 ± 0.9724
Tuned Baseline  NRMSE : 1.5311 ± 0.0553

--- log1p scale ---
Tuned Baseline  R²    : 0.8894 ± 0.0023
Tuned Baseline  MAE   : 0.3648 ± 0.0045
Tuned B

In [22]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Save Results                            #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

RESULTS_PATH = "data/results/nn_results.csv"
os.makedirs(os.path.dirname(RESULTS_PATH), exist_ok=True)

results_df = pd.DataFrame(run_results)

if os.path.exists(RESULTS_PATH):
    results_df.to_csv(RESULTS_PATH, mode="a", header=False, index=False)
    print(f"Appended {len(results_df)} rows to {RESULTS_PATH}")
else:
    results_df.to_csv(RESULTS_PATH, index=False)
    print(f"Created {RESULTS_PATH} with {len(results_df)} rows")

print(pd.read_csv(RESULTS_PATH).tail(len(results_df)).to_string(index=False))


Appended 5 rows to data/results/nn_results.csv
          timestamp          model  n_layers  width  learning_rate  batch_size  seed  val_loss  zero_demand_count  mean_actual       r2      mae      rmse    nrmse
2026-06-27T15:37:26 baseline_tuned         2     64        0.00005         256     0  0.193888            15505.0    17.569093 0.792809 7.162975 25.548620 1.454180
2026-06-27T15:37:26 baseline_tuned         2     64        0.00005         256     1  0.197139            15505.0    17.569093 0.779975 7.322914 26.328013 1.498541
2026-06-27T15:37:26 baseline_tuned         2     64        0.00005         256     2  0.190834            15505.0    17.569093 0.745487 7.586333 28.316288 1.611710
2026-06-27T15:37:26 baseline_tuned         2     64        0.00005         256     3  0.193196            15505.0    17.569093 0.774283 7.370461 26.666344 1.517799
2026-06-27T15:37:26 baseline_tuned         2     64        0.00005         256     4  0.198949            15505.0    17.569093 0.7995

## Deeper Model — Training

Run `ARCH_DEEPER = (4 hidden layers, 64 units)` across all seeds.
Width is fixed; depth doubles relative to the baseline.


In [23]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Deeper Model — Training                 #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

deeper_val_losses = []
deeper_models     = []

for seed in SEEDS:
    model, val_loss = train_model(
        ARCH_DEEPER, X_train, y_train, X_val, y_val,
        hex_train, hex_val,
        seed=seed, device=device,
        lr=BEST_LR.get(ARCH_DEEPER, config.LEARNING_RATE),
    )
    deeper_val_losses.append(val_loss)
    deeper_models.append(model)
    print(f"  seed={seed}  val_loss={val_loss:.4f}")

print(f"\nDeeper  val loss:  {np.mean(deeper_val_losses):.4f} ± {np.std(deeper_val_losses):.4f}")


  epoch   1/100  train=1.0366  val=0.5629 *
  epoch   2/100  train=0.3367  val=0.3585 *
  epoch   3/100  train=0.2540  val=0.3120 *
  epoch   4/100  train=0.2276  val=0.2864 *
  epoch   5/100  train=0.2129  val=0.2691 *
  epoch   6/100  train=0.2031  val=0.2605 *
  epoch   7/100  train=0.1957  val=0.2475 *
  epoch   8/100  train=0.1898  val=0.2465 *
  epoch   9/100  train=0.1849  val=0.2358 *
  epoch  10/100  train=0.1809  val=0.2305 *
  epoch  11/100  train=0.1774  val=0.2304 *
  epoch  12/100  train=0.1743  val=0.2226 *
  epoch  13/100  train=0.1717  val=0.2263 (no improvement 1/10)
  epoch  14/100  train=0.1695  val=0.2210 *
  epoch  15/100  train=0.1675  val=0.2174 *
  epoch  16/100  train=0.1657  val=0.2154 *
  epoch  17/100  train=0.1642  val=0.2150 *
  epoch  18/100  train=0.1629  val=0.2124 *
  epoch  19/100  train=0.1616  val=0.2100 *
  epoch  20/100  train=0.1606  val=0.2103 (no improvement 1/10)
  epoch  21/100  train=0.1596  val=0.2083 *
  epoch  22/100  train=0.1587  val=0

## Deeper Model — Evaluation

Evaluate on the held-out **test set** using the best-checkpoint model from each
seed, then report mean ± std across seeds.

Metrics:
- **Zero-demand rows** — share of test rows with `trip_count = 0` (context for inflated R²)
- **Mean Actual Test Demand** — average true trip count in the test set
- **R²** — coefficient of determination (share of variance explained)
- **MAE** — mean absolute error (interpretable in trip counts)
- **RMSE** — root mean squared error (penalises large misses more)
- **NRMSE** — RMSE normalised by mean actual demand (scale-free)

In [24]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Deeper   Model — Evaluation               #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

import datetime

X_test_t   = torch.as_tensor(X_test,   dtype=torch.float32).to(device)
hex_test_t = torch.as_tensor(hex_test, dtype=torch.long).to(device)

run_results  = []
r2_scores, mae_scores, rmse_scores, nrmse_scores = [], [], [], []
n_layers, width = ARCH_DEEPER

zero_demand_count = int((y_test == 0).sum())
zero_demand_pct   = zero_demand_count / len(y_test) * 100
mean_actual       = float(y_test.mean())
ss_tot            = float(np.sum((y_test - mean_actual) ** 2))

print(f"Zero-demand rows: {zero_demand_count:,} / {len(y_test):,} ({zero_demand_pct:.1f}%)")
print(f"Mean actual test demand : {mean_actual:.4f}\n")

for seed, model in zip(SEEDS, deeper_models):
    model.eval()
    with torch.no_grad():
        preds = model(X_test_t, hex_test_t).cpu().numpy()
        preds = np.expm1(preds)

    mae   = float(np.mean(np.abs(preds - y_test)))
    rmse  = float(np.sqrt(np.mean((preds - y_test) ** 2)))
    nrmse = rmse / mean_actual if mean_actual > 0 else float("nan")
    r2    = 1.0 - float(np.sum((preds - y_test) ** 2)) / ss_tot

    r2_scores.append(r2)
    mae_scores.append(mae)
    rmse_scores.append(rmse)
    nrmse_scores.append(nrmse)

    print(f"  seed={seed}  R²={r2:.4f}  MAE={mae:.4f}  RMSE={rmse:.4f}  NRMSE={nrmse:.4f}")

    run_results.append({
        "timestamp"         : datetime.datetime.now().isoformat(timespec="seconds"),
        "model"             : ARCH_NAMES.get((n_layers, width), "unknown"),
        "n_layers"          : n_layers,
        "width"             : width,
        "learning_rate"     : BEST_LR.get(ARCH_DEEPER, config.LEARNING_RATE),
        "batch_size"        : BATCH_SIZE,
        "seed"              : seed,
        "val_loss"          : deeper_val_losses[seed],
        "zero_demand_count" : zero_demand_count,
        "zero_demand_pct"   : round(zero_demand_pct, 2),
        "mean_actual"       : round(mean_actual, 6),
        "r2"                : round(r2,    6),
        "mae"               : round(mae,   6),
        "rmse"              : round(rmse,  6),
        "nrmse"             : round(nrmse, 6),
    })

arch_label = ARCH_NAMES.get((n_layers, width), "model")
print(f"\n{arch_label.capitalize()}  R²    : {np.mean(r2_scores):.4f} ± {np.std(r2_scores):.4f}")
print(f"{arch_label.capitalize()}  MAE   : {np.mean(mae_scores):.4f} ± {np.std(mae_scores):.4f}")
print(f"{arch_label.capitalize()}  RMSE  : {np.mean(rmse_scores):.4f} ± {np.std(rmse_scores):.4f}")
print(f"{arch_label.capitalize()}  NRMSE : {np.mean(nrmse_scores):.4f} ± {np.std(nrmse_scores):.4f}")

Zero-demand rows: 15,505 / 43,362 (35.8%)
Mean actual test demand : 17.5691

  seed=0  R²=0.7984  MAE=7.2243  RMSE=25.1988  NRMSE=1.4343
  seed=1  R²=0.7962  MAE=7.2032  RMSE=25.3359  NRMSE=1.4421
  seed=2  R²=0.7851  MAE=7.3068  RMSE=26.0171  NRMSE=1.4808
  seed=3  R²=0.7644  MAE=7.5120  RMSE=27.2419  NRMSE=1.5506
  seed=4  R²=0.7914  MAE=7.1793  RMSE=25.6373  NRMSE=1.4592

Deep  R²    : 0.7871 ± 0.0122
Deep  MAE   : 7.2851 ± 0.1213
Deep  RMSE  : 25.8862 ± 0.7338
Deep  NRMSE : 1.4734 ± 0.0418


In [25]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Save Results                            #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

RESULTS_PATH = "data/results/nn_results.csv"
os.makedirs(os.path.dirname(RESULTS_PATH), exist_ok=True)

results_df = pd.DataFrame(run_results)

if os.path.exists(RESULTS_PATH):
    results_df.to_csv(RESULTS_PATH, mode="a", header=False, index=False)
    print(f"Appended {len(results_df)} rows to {RESULTS_PATH}")
else:
    results_df.to_csv(RESULTS_PATH, index=False)
    print(f"Created {RESULTS_PATH} with {len(results_df)} rows")

print(pd.read_csv(RESULTS_PATH).tail(len(results_df)).to_string(index=False))


Appended 5 rows to data/results/nn_results.csv


ParserError: Error tokenizing data. C error: Expected 14 fields in line 62, saw 15


## Hyperparameter Search — Deeper Model

Same grid search as for the baseline, run on `ARCH_DEEPER = (4 hidden layers, 64 units)`.
Result is stored in `BEST_HP[ARCH_DEEPER]`.

| Hyperparameter | Candidates |
|---|---|
| `lr`           | 2e-5, 3e-5, 4e-5, 5e-5, 6e-5, 7e-5, 1e-4 |

Full grid: 4 × 4 × 3 = **48 combinations** × 1 seed = **48 training runs**.  
The best combination is stored in `BEST_HP[ARCH_DEEPER]`.

In [28]:
run_hp_search(
    ARCH_DEEPER,
    X_train, y_train, X_val, y_val,
    hex_train, hex_val,
    device=device,
)


HP Search — deep
7 combos × 1 seed(s) = 7 runs  |  patience=10

   #        lr   dropout        wd    mean val       std
--------------------------------------------------
   1     2e-05      0.00   0.0e+00      0.1986    0.0000
   2     3e-05      0.00   0.0e+00      0.1982    0.0000
   3     4e-05      0.00   0.0e+00      0.1958    0.0000
   4     5e-05      0.00   0.0e+00      0.1950    0.0000
   5     6e-05      0.00   0.0e+00      0.1949    0.0000
   6     7e-05      0.00   0.0e+00      0.1947    0.0000
   7     1e-04      0.00   0.0e+00      0.1970    0.0000

--- Top 5 ---
     lr  dropout  weight_decay  mean_val_loss  std_val_loss
0.00007      0.0           0.0       0.194685           0.0
0.00006      0.0           0.0       0.194947           0.0
0.00005      0.0           0.0       0.194965           0.0
0.00004      0.0           0.0       0.195832           0.0
0.00010      0.0           0.0       0.197018           0.0

BEST_HP[deep] = {'lr': 7e-05, 'dropout': 0.0, 'weight

{'lr': 7e-05, 'dropout': 0.0, 'weight_decay': 0.0}

## LR Search — Wider Model

Same grid search as for the baseline, run on `ARCH_WIDER = (2 hidden layers, 128 units)`.
Result is stored in `BEST_LR[ARCH_WIDER]`.


In [27]:
run_lr_search(ARCH_WIDER,    X_train, y_train, X_val, y_val, hex_train, hex_val, device=device)  # (2, 128)


NameError: name 'run_lr_search' is not defined

## Wider Model — Training

Run `ARCH_WIDER = (2 hidden layers, 128 units)` across all seeds.
Depth is fixed; width doubles relative to the baseline.


In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Wider Model — Training                  #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

wider_val_losses = []
wider_models     = []

for seed in SEEDS:
    model, val_loss = train_model(
        ARCH_WIDER, X_train, y_train, X_val, y_val,
        hex_train, hex_val,
        seed=seed, device=device,
        lr=BEST_LR.get(ARCH_WIDER, config.LEARNING_RATE),
    )
    wider_val_losses.append(val_loss)
    wider_models.append(model)
    print(f"  seed={seed}  val_loss={val_loss:.4f}")

print(f"\nWider  val loss:  {np.mean(wider_val_losses):.4f} ± {np.std(wider_val_losses):.4f}")


## Wider Model — Evaluation

Evaluate on the held-out **test set** using the best-checkpoint model from each
seed, then report mean ± std across seeds.

Metrics:
- **Zero-demand rows** — share of test rows with `trip_count = 0` (context for inflated R²)
- **Mean Actual Test Demand** — average true trip count in the test set
- **R²** — coefficient of determination (share of variance explained)
- **MAE** — mean absolute error (interpretable in trip counts)
- **RMSE** — root mean squared error (penalises large misses more)
- **NRMSE** — RMSE normalised by mean actual demand (scale-free)

In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Wider    Model — Evaluation               #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

import datetime

X_test_t   = torch.as_tensor(X_test,   dtype=torch.float32).to(device)
hex_test_t = torch.as_tensor(hex_test, dtype=torch.long).to(device)

run_results  = []
r2_scores, mae_scores, rmse_scores, nrmse_scores = [], [], [], []
n_layers, width = ARCH_WIDER

zero_demand_count = int((y_test == 0).sum())
zero_demand_pct   = zero_demand_count / len(y_test) * 100
mean_actual       = float(y_test.mean())
ss_tot            = float(np.sum((y_test - mean_actual) ** 2))

print(f"Zero-demand rows: {zero_demand_count:,} / {len(y_test):,} ({zero_demand_pct:.1f}%)")
print(f"Mean actual test demand : {mean_actual:.4f}\n")

for seed, model in zip(SEEDS, wider_models):
    model.eval()
    with torch.no_grad():
        preds = model(X_test_t, hex_test_t).cpu().numpy()
        preds = np.expm1(preds)

    mae   = float(np.mean(np.abs(preds - y_test)))
    rmse  = float(np.sqrt(np.mean((preds - y_test) ** 2)))
    nrmse = rmse / mean_actual if mean_actual > 0 else float("nan")
    r2    = 1.0 - float(np.sum((preds - y_test) ** 2)) / ss_tot

    r2_scores.append(r2)
    mae_scores.append(mae)
    rmse_scores.append(rmse)
    nrmse_scores.append(nrmse)

    print(f"  seed={seed}  R²={r2:.4f}  MAE={mae:.4f}  RMSE={rmse:.4f}  NRMSE={nrmse:.4f}")

    run_results.append({
        "timestamp"         : datetime.datetime.now().isoformat(timespec="seconds"),
        "model"             : ARCH_NAMES.get((n_layers, width), "unknown"),
        "n_layers"          : n_layers,
        "width"             : width,
        "learning_rate"     : BEST_LR.get(ARCH_WIDER, config.LEARNING_RATE),
        "batch_size"        : BATCH_SIZE,
        "seed"              : seed,
        "val_loss"          : wider_val_losses[seed],
        "zero_demand_count" : zero_demand_count,
        "zero_demand_pct"   : round(zero_demand_pct, 2),
        "mean_actual"       : round(mean_actual, 6),
        "r2"                : round(r2,    6),
        "mae"               : round(mae,   6),
        "rmse"              : round(rmse,  6),
        "nrmse"             : round(nrmse, 6),
    })

arch_label = ARCH_NAMES.get((n_layers, width), "model")
print(f"\n{arch_label.capitalize()}  R²    : {np.mean(r2_scores):.4f} ± {np.std(r2_scores):.4f}")
print(f"{arch_label.capitalize()}  MAE   : {np.mean(mae_scores):.4f} ± {np.std(mae_scores):.4f}")
print(f"{arch_label.capitalize()}  RMSE  : {np.mean(rmse_scores):.4f} ± {np.std(rmse_scores):.4f}")
print(f"{arch_label.capitalize()}  NRMSE : {np.mean(nrmse_scores):.4f} ± {np.std(nrmse_scores):.4f}")

In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Save Results                            #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

RESULTS_PATH = "data/results/nn_results.csv"
os.makedirs(os.path.dirname(RESULTS_PATH), exist_ok=True)

results_df = pd.DataFrame(run_results)

if os.path.exists(RESULTS_PATH):
    results_df.to_csv(RESULTS_PATH, mode="a", header=False, index=False)
    print(f"Appended {len(results_df)} rows to {RESULTS_PATH}")
else:
    results_df.to_csv(RESULTS_PATH, index=False)
    print(f"Created {RESULTS_PATH} with {len(results_df)} rows")

print(pd.read_csv(RESULTS_PATH).tail(len(results_df)).to_string(index=False))


## Hyperparameter Search — Wider Model

Same grid search as for the baseline, run on `ARCH_WIDER = (2 hidden layers, 128 units)`.
Result is stored in `BEST_HP[ARCH_WIDER]`.

| Hyperparameter | Candidates |
|---|---|
| `lr`           | 2e-5, 3e-5, 4e-5, 5e-5, 6e-5, 7e-5, 1e-4 |

Full grid: 4 × 4 × 3 = **48 combinations** × 1 seed = **48 training runs**.  
The best combination is stored in `BEST_HP[ARCH_WIDER]`.

In [ ]:
run_hp_search(
    ARCH_WIDER,
    X_train, y_train, X_val, y_val,
    hex_train, hex_val,
    device=device,
)
